# hallu_hierarchy — M1.5 (correctness fixes, no GPU)
Reruns only `--stages correctness` on saved M1 outputs. Settings: Accelerator **None**, Internet on.
Add the Kaggle Dataset holding the M1 outputs (generation/, data/, correctness/) as an input.
Manual decisions go in `hallu_hierarchy/review/<run_name>_overrides.csv` in the repo (see correctness/review.py).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

if hf := get_secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = hf
gh = get_secret('GITHUB_TOKEN')
REPO = 'github.com/harshith20/Long_Range_MASE.git'
url = f'https://{gh}@{REPO}' if gh else f'https://{REPO}'
os.system(f'git clone -q {url} /kaggle/working/repo')
os.chdir('/kaggle/working/repo')
!git log -1 --oneline

In [ ]:
!pip install -q pyyaml pytest
!python -m pytest -q hallu_hierarchy/tests

In [ ]:
CONFIG = 'configs/qwen3b_popqa_dev.yaml'
INPUT = '/kaggle/input/<your-m1-outputs-dataset>'   # folder containing the run (or outputs/<run_name>)
!python -m hallu_hierarchy.run --config {CONFIG} --stages correctness --input-run-dir {INPUT}

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display

run = sorted(glob.glob('/kaggle/working/outputs/*/correctness'))[-1]
display(pd.read_csv(f'{run}/review/disagreements.csv')[['qid','relation','question','pred','aliases','matched_alias_relaxed']])
display(pd.read_csv(f'{run}/abstain_log.csv').pattern.value_counts())  # audit refusal matches
for png in sorted(glob.glob('/kaggle/working/outputs/*/plots/*.png')):
    display(Image(png))